In [34]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
import ast
import numpy as np

In [17]:
# read market news data
df = pd.read_csv('market_news_data.csv')
df.head(5)

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_55735/2321408899.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('market_news_data.csv')


,timestamp,symbols,headline,summary
0,2025-12-31T22:30:46Z,['NVDA'],If You Invested $100 In NVIDIA Stock 5 Years A...,
1,2025-12-31T21:21:36Z,"['SPY', 'VGK']",'U.S. Finds Ukraine Didn't Target Putin in Dro...,NaN
2,2025-12-31T21:06:12Z,"['AAPL', 'AOS', 'DASH', 'DG', 'ET', 'IP', 'RAC...","Winners, Losers, Wild Opinions: Benzinga Chat,...",2025 was a rollercoaster for Benzinga chat roo...
3,2025-12-31T20:15:07Z,"['SPY', 'TSM']",TSMC Says U.S. Department Of Commerce Has Gran...,NaN
4,2025-12-31T20:59:34Z,['SPY'],President Trump Posts On Truth Social 'We are ...,NaN


In [18]:
df.tail(5)

,timestamp,symbols,headline,summary
168613,2015-01-02T14:53:08Z,"['BROAD', 'FED', 'SPY']",Cleveland Fed Pres Mester Recently Speaking on...,NaN
168614,2015-01-02T14:09:15Z,['AAPL'],Argus Expects Apple To Report Strong Q1 Result...,NaN
168615,2015-01-02T10:01:01Z,"['EWC', 'FXI', 'SPY']","Economic Calendar for Friday January 2, 2015",NaN
168616,2015-01-01T19:17:33Z,['KO'],Mike Khouw Sees Unusual Options Activity In Th...,NaN
168617,2015-01-01T19:14:12Z,"['AAL', 'AAPL', 'IBM', 'INTC', 'WYNN']",CNBC's Biggest Pops & Drops of 2014,NaN


In [19]:
# 1. Setup Device
device = "mps" # Apple Silicon

# 2. Load FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create the sentiment pipeline
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

# 3. Prepar text for FinBERT
def combine_text(row):
    headline = str(row['headline'])
    summary = str(row['summary'])
    
    # If summary is 'nan', 'NaN', or empty, just use the headline
    if not summary or summary.lower() == 'nan' or len(summary.strip()) == 0:
        return headline
    
    # Otherwise, join them (FinBERT handles up to 512 tokens)
    return f"{headline}. {summary}"

processed_text = df.apply(combine_text, axis=1).tolist()

# 4. Batch processing
batch_size = 64
results = []

for i in tqdm(range(0, len(processed_text), batch_size)):
    batch = processed_text[i : i + batch_size]
    # Truncation is critical here since adding summaries makes the text longer
    batch_results = nlp(batch, truncation=True, max_length=512)
    results.extend(batch_results)

# 5. Extract results back to DataFrame
df['sentiment_label'] = [res['label'] for res in results]
df['sentiment_score'] = [res['score'] for res in results]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

100%|██████████| 2635/2635 [26:19<00:00,  1.67it/s]


In [20]:
df.head(5)

,timestamp,symbols,headline,summary,sentiment_label,sentiment_score
0,2025-12-31T22:30:46Z,['NVDA'],If You Invested $100 In NVIDIA Stock 5 Years A...,,neutral,0.899845
1,2025-12-31T21:21:36Z,"['SPY', 'VGK']",'U.S. Finds Ukraine Didn't Target Putin in Dro...,NaN,neutral,0.712345
2,2025-12-31T21:06:12Z,"['AAPL', 'AOS', 'DASH', 'DG', 'ET', 'IP', 'RAC...","Winners, Losers, Wild Opinions: Benzinga Chat,...",2025 was a rollercoaster for Benzinga chat roo...,neutral,0.620395
3,2025-12-31T20:15:07Z,"['SPY', 'TSM']",TSMC Says U.S. Department Of Commerce Has Gran...,NaN,positive,0.866344
4,2025-12-31T20:59:34Z,['SPY'],President Trump Posts On Truth Social 'We are ...,NaN,neutral,0.579338


In [22]:
df.to_csv("market_news_with_sentiment.csv",index=False)

In [ ]:
# Read market new sentiment data
df = pd.read_csv("market_news_with_sentiment.csv")
df["symbols"] = df["symbols"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_exploded = df.explode("symbols").rename(columns={"symbols": "ticker"})

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_55735/1997736628.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("market_news_with_sentiment.csv")


In [25]:
# read stock and etf tickers
stock_returns = pd.read_csv("stock_returns.csv")
etf_returns = pd.read_csv("etf_returns.csv")

stock_returns["date"] = pd.to_datetime(stock_returns["date"])

etf_returns["date"] = pd.to_datetime(etf_returns["date"])

tickers = stock_returns.columns.tolist() + etf_returns.columns.tolist()

In [ ]:
# remove excess tickers
target_tickers = set(tickers)
df_exploded = df_exploded[df_exploded["ticker"].isin(target_tickers)].copy()
df_exploded["ticker"].unique()

array(['NVDA', 'SPY', 'AAPL', 'XOM', 'AMZN', 'JNJ', 'MSFT', 'XLE', 'XLK',
       'QQQ', 'KO', 'CAT', 'JPM', 'V', 'VNQ', 'XLV', 'IWM', 'TLT', 'EEM',
       'EFA'], dtype=object)

In [27]:
# monthly rebalance
df_exploded["timestamp"] = pd.to_datetime(df_exploded["timestamp"], utc=True)

df_exploded["month"] = (
    df_exploded["timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

monthly_sentiment = (
    df_exploded
    .groupby(["month", "ticker"])["sentiment_score"]
    .mean()
    .reset_index()
)

sentiment_matrix = monthly_sentiment.pivot(
    index="month",
    columns="ticker",
    values="sentiment_score"
).sort_index()

sentiment_matrix = sentiment_matrix.reindex(columns=tickers)

# fill missing values with 0 (neutral sentiment)
sentiment_matrix = sentiment_matrix.fillna(0)

# check
set(sentiment_matrix.columns) == set(tickers)

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_55735/4212619496.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


True

In [33]:
sentiment_matrix = sentiment_matrix.iloc[1:]
sentiment_matrix.head()

ticker,date,AAPL,AMZN,CAT,JNJ,JPM,KO,MSFT,NVDA,V,...,EEM,EFA,IWM,QQQ,SPY,TLT,VNQ,XLE,XLK,XLV
month,,,,,,,,,,,,,,,,,,,,,
2015-01-31,0.0,0.824090,0.825504,0.842298,0.816898,0.801158,0.834099,0.838305,0.907710,0.817808,...,0.943040,0.0,0.865964,0.907663,0.805798,0.908362,0.897833,0.869395,0.000000,0.000000
2015-02-28,0.0,0.839402,0.837314,0.809278,0.802339,0.835376,0.859523,0.815664,0.799796,0.817021,...,0.868581,0.0,0.752417,0.799046,0.770433,0.771994,0.827680,0.908076,0.913162,0.000000
2015-03-31,0.0,0.831748,0.847928,0.909503,0.849546,0.780087,0.843417,0.829032,0.816644,0.818553,...,0.897726,0.0,0.878679,0.881801,0.824404,0.805766,0.742166,0.829155,0.904377,0.895456
2015-04-30,0.0,0.815684,0.835431,0.822454,0.819907,0.854351,0.833677,0.819139,0.879603,0.793180,...,0.903760,0.0,0.908198,0.867923,0.782319,0.910253,0.745634,0.857838,0.686085,0.810549
2015-05-31,0.0,0.834582,0.826630,0.847653,0.808938,0.816704,0.847786,0.827964,0.784542,0.867394,...,0.935117,0.0,0.817752,0.710503,0.820806,0.909713,0.810014,0.751347,0.000000,0.000000


In [35]:
# Q = alpha *(Sentiment Matrix - 0.5) to center sentiment around 0
alpha = 0.1
Q_matrix = alpha * (sentiment_matrix - 0.5)

In [41]:
Q_matrix.head()

ticker,date,AAPL,AMZN,CAT,JNJ,JPM,KO,MSFT,NVDA,V,...,EEM,EFA,IWM,QQQ,SPY,TLT,VNQ,XLE,XLK,XLV
month,,,,,,,,,,,,,,,,,,,,,
2015-01-31,-0.05,0.032409,0.032550,0.034230,0.031690,0.030116,0.033410,0.033830,0.040771,0.031781,...,0.044304,-0.05,0.036596,0.040766,0.030580,0.040836,0.039783,0.036939,-0.050000,-0.050000
2015-02-28,-0.05,0.033940,0.033731,0.030928,0.030234,0.033538,0.035952,0.031566,0.029980,0.031702,...,0.036858,-0.05,0.025242,0.029905,0.027043,0.027199,0.032768,0.040808,0.041316,-0.050000
2015-03-31,-0.05,0.033175,0.034793,0.040950,0.034955,0.028009,0.034342,0.032903,0.031664,0.031855,...,0.039773,-0.05,0.037868,0.038180,0.032440,0.030577,0.024217,0.032916,0.040438,0.039546
2015-04-30,-0.05,0.031568,0.033543,0.032245,0.031991,0.035435,0.033368,0.031914,0.037960,0.029318,...,0.040376,-0.05,0.040820,0.036792,0.028232,0.041025,0.024563,0.035784,0.018609,0.031055
2015-05-31,-0.05,0.033458,0.032663,0.034765,0.030894,0.031670,0.034779,0.032796,0.028454,0.036739,...,0.043512,-0.05,0.031775,0.021050,0.032081,0.040971,0.031001,0.025135,-0.050000,-0.050000


In [37]:
# function to get specifc view for a given month and ticker
def get_Qt(Q_matrix, date, ticker):
    return Q_matrix.loc[date, ticker].values.reshape(-1,1)